# Imports and Global Constants
## This cell imports all necessary libraries and defines global constants used throughout the script.

In [ ]:
# %%
import datetime # For generating unique filenames based on timestamp
import json     # For reading from and writing to JSON files
import time     # For pausing execution (e.g., waiting for page loads)
from os import path, makedirs # For path manipulation and directory creation
from typing import Literal # For type hinting, though 'Web' is used from selenium itself

import pandas as pd # For reading game IDs from a CSV file
from selenium import webdriver # The core Selenium WebDriver library
from selenium.webdriver.common.by import By # For specifying element selection strategies (e.g., By.CSS_SELECTOR, By.CLASS_NAME)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Firefox specific imports ---
from selenium.webdriver.firefox.service import Service as FirefoxService # Service object for Firefox WebDriver
from selenium.webdriver.remote.webdriver import WebDriver as Web # Type hint for WebDriver instances
from webdriver_manager.firefox import GeckoDriverManager # Automatically downloads and manages geckodriver (for Firefox)

# --- Global Constants ---
# Directory where output JSON files will be saved
OUTPUT_DIR = "output"
LANGUAGES = {
    "english": {
        "game_base_url": "https://www.espn.com/mlb/",
        "scoreboard_base_url": "https://www.espn.com/mlb/scoreboard/",
        "date_param": "_/date/"
    },
    "spanish": {
        "game_base_url": "https://espndeportes.espn.com/beisbol/mlb/",
        "scoreboard_base_url": "https://espndeportes.espn.com/beisbol/mlb/resultados/",
        "date_param": "_/fecha/"
    }, 
    "dutch": {
        "game_base_url": "https://www.espn.nl/mlb/",
        "scoreboard_base_url": "https://www.espn.nl/mlb/scorebord/",
        "date_param": "_/date/"
    }
}
GAMEID_FILE = "game_ids.csv"  # CSV file containing game IDs to scrape
print("Imports and global constants loaded successfully!")
# %%

# Utility Function: wait_el_text
#### This function is a crucial utility for robust web scraping. 
#### It waits until a specified web element's text content is not empty, indicating that the content has likely loaded on the page. 
#### This prevents errors that can occur when trying to extract text from an element before the JavaScript has populated it.


In [2]:
def wait_el_text(driver, selector: By, el_path: str, interval: float = 2, timeout: float= 10):
    """Returns element text content when it is loaded, 
    this ensures desired content is loaded and ready for extraction.

    Args:
        driver (_type_): web driver
        url (str): url to get
        selector (By): selector used for driver.find_element
        el_path (str): path to the element to wait on
        interval (float): interval to retry getting the text of element
        timeout (float): max time for waiting the element
        
    Raises:
        TimeoutError: if element text not loaded after sepcified timeout
    """
    time_passed = 0
    
    while time_passed < timeout:
        el = driver.find_element(selector,el_path)
        if el.text != "":
            return el.text
        
        time.sleep(interval)
        time_passed += interval
        
    raise TimeoutError(f"Timeout while waiting element at: {el_path} to load")

# Utility Function: append_json
#### This function handles saving data to a JSON file. 
#### It's designed to append new data to an existing JSON array if the file already contains one, or to create a new file/list if it doesn't exist or is improperly formatted.



In [ ]:
# %%
def append_json(file_name: str, data: list):
    """
    Appends new data to a JSON file. If the file exists and contains a JSON array,
    the new data is extended to that array. If the file is empty, invalid, or
    doesn't exist, a new JSON array is created with the provided data.

    Args:
        file_name (str): The name of the JSON file to which data will be appended.
                         This file will be saved in the `OUTPUT_DIR`.
        data (list or dict): The data to append. If a list of dictionaries, it's extended.
                             If a single dictionary, it's appended as one item to the list.
    """
    new_data = [] # Initialize an empty list to hold content before writing
    full_path = path.join(OUTPUT_DIR, file_name) # Construct the full path to the output file
    
    try:
        # Attempt to open and load existing JSON content
        with open(full_path, "r") as file:
            existing_content = json.load(file)
            # If the existing content is a list, use it as the base for appending
            if isinstance(existing_content, list):
                new_data = existing_content
            else:
                # If existing content is a single object (e.g., dict), wrap it in a list
                # This ensures we always work with a list structure for appending
                new_data = [existing_content]
    except (json.JSONDecodeError, FileNotFoundError):
        # If the file is empty, contains invalid JSON, or doesn't exist,
        # we start with an empty list for `new_data`. The provided `data` will be the first entry.
        print(f"File '{file_name}' not found or invalid JSON. Creating new file/content.")
        new_data = [] # Ensure new_data is an empty list to start fresh

    # Append the incoming 'data' to our `new_data` list
    if isinstance(data, list):
        new_data.extend(data) # If 'data' is already a list, extend the existing list
    else:
        new_data.append(data) # If 'data' is a single item (e.g., a dict), append it

    # Write the updated (or new) content back to the JSON file
    with open(full_path, "w") as file:
        json.dump(new_data, file, indent=4) # Use indent=4 for pretty-printing JSON

print("`append_json` function defined.")
# %%

# Utility Function: get_game_ids_for_date
#### This function is responsible for navigating to the daily scoreboard page and extracting the game IDs for all games played on a specific date for a given language.

In [ ]:
# %%
def get_game_ids_for_date(driver, scoreboard_base_url, date_param, date_str):
    """
    Navigates to the ESPN MLB scoreboard page for a specific date and language,
    and extracts all unique game IDs from the game sections displayed.

    Args:
        driver (Web): The Selenium WebDriver instance.
        scoreboard_base_url (str): The base URL for the scoreboard page (e.g., "https://www.espn.com/mlb/scoreboard/" or "https://espndeportes.espn.com/beisbol/mlb/resultados/").
        date_param (str): The URL segment for the date parameter (e.g., "_/date/" for English, "_/fecha/" for Spanish).
        date_str (str): The date in YYYYMMDD format (e.g., "20250622").

    Returns:
        List[str]: A list of extracted unique game IDs (strings). Returns an empty list if no games are found or an error occurs.
    """
    # Construct the full URL for the scoreboard page for the given date and language
    full_url = f"{scoreboard_base_url}{date_param}{date_str}"
    print(f"Attempting to get game IDs from: {full_url}")
    
    driver.get(full_url) # Navigate the browser to the scoreboard URL
    
    game_ids = [] # Initialize an empty list to store found game IDs
    try:
        # Use WebDriverWait to explicitly wait until at least one scoreboard section
        # with an ID starting with '40' (typical ESPN game ID prefix) is present.
        # This makes the script more robust to page loading times.
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "section.Scoreboard[id^='40']"))
        )
        
        # Once at least one such element is present, find all game sections.
        # These elements have the class "Scoreboard" and their 'id' attribute contains the game ID.
        game_sections = driver.find_elements(By.CSS_SELECTOR, "section.Scoreboard[id^='40']")
        
        # Iterate through each found game section and extract its 'id' attribute
        for section in game_sections:
            game_id = section.get_attribute("id")
            if game_id: # Ensure the id attribute is not empty
                game_ids.append(game_id)
        
        print(f"Found {len(game_ids)} games for {date_str} at {scoreboard_base_url}.")
        
    except Exception as e:
        # Catch any exceptions during the process (e.g., TimeoutError if no games load)
        print(f"No games found or error loading page for {date_str} at {scoreboard_base_url}: {e}")
    
    # Return a list of unique game IDs (using set to remove duplicates, then converting back to list)
    return list(set(game_ids))

print("`get_game_ids_for_date` function defined.")
# %%

# MLB Play-By-Play Scraper: MLB_play_by_play
#### This function navigates to the ESPN play-by-play page for a given game ID and extracts the chronological narrative of the game.

In [ ]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException # Import NoSuchElementException

def MLB_play_by_play(driver: selenium.webdriver.remote.webdriver.WebDriver, game_id: str, game_base_url: str) -> str:
    """
    Navigates to the ESPN MLB play-by-play page for a given game ID and
    extracts the entire play-by-play script into a single concatenated string,
    including inning information and, if available, scores for each play,
    and initial home/away team names.

    Args:
        driver (Web): The Selenium WebDriver instance to use for navigation and extraction.
        game_id (str): The unique identifier for the MLB game (e.g., "401579246").
        game_base_url (str): The base ESPN MLB URL for game-specific pages (e.g., "https://www.espn.com/mlb" or "https://www.espn.com/es/mlb").

    Returns:
        str: A single string containing the full play-by-play narrative for the game,
             with sentences punctuated correctly and including inning context.
             Returns an empty string if data cannot be retrieved.
    """
    print(f"Extracting play-by-play: {game_id} (URL: {game_base_url}playbyplay/_/gameId/{game_id})")
    play_url = f"{game_base_url}playbyplay/_/gameId/{game_id}"
    inning_label_class_name = "HalfInningHeader__period"
    play_description_class_name = "PlayHeader__description"
    
    # These are likely for scores, not full team names, but we'll extract them if present.
    away_score_class_name = "PlayHeader__score PlayHeader__score--away"
    home_score_class_name = "PlayHeader__score PlayHeader__score--home"
    
    # Class for actual team names, usually found in a scoreboard or game header.
    # This is a common class on ESPN. You might need to adjust this.
    team_name_class = "HalfInningHeader__name"

    driver.get(play_url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, inning_label_class_name))
        )
    except TimeoutException:
        print(f"Timeout waiting for inning elements for game {game_id} at {game_base_url}. Skipping.")
        return ""

    joined_text = []

    # --- Attempt to get Home and Away Team Names (Full names, once at the top) ---
    home_team_name = "Home Team"
    away_team_name = "Away Team"
    try:
        # Find all elements that are likely team names in the scoreboard/header
        team_name_elements = driver.find_elements(By.CLASS_NAME, team_name_class)
        if len(team_name_elements) >= 2:
            # Assuming the first two are usually Away and Home team names respectively
            away_team_name = team_name_elements[0].text.strip()
            home_team_name = team_name_elements[1].text.strip()
            joined_text.append(f"Game between {away_team_name} and {home_team_name}.")
            print(f"Detected Teams: Away - {away_team_name}, Home - {home_team_name}")
        else:
            joined_text.append(f"Game between {away_team_name} and {home_team_name} (Team names not explicitly found).")
            print(f"Could not reliably determine team names for game {game_id} using class '{team_name_class}'.")
    except Exception as e:
        joined_text.append(f"Game between {away_team_name} and {home_team_name} (Error getting team names).")
        print(f"Error trying to get actual team names: {e}")

    # Find all inning containers
    inning_containers = driver.find_elements(By.CLASS_NAME, "HalfInning")

    if not inning_containers:
        print(f"No inning containers found for game {game_id}. Falling back to just play descriptions.")
        # Fallback if inning structure is not as expected
        play_elements = driver.find_elements(By.CLASS_NAME, play_description_class_name)
        for play_div in play_elements:
            play_text = play_div.text.strip()
            if play_text:
                if not play_text.endswith(('.', '!', '?')):
                    play_text += "."
                joined_text.append(play_text)
        return "\n".join(joined_text).strip()

    for container in inning_containers:
        current_inning = "Unknown Inning"
        try:
            inning_label_element = container.find_element(By.CLASS_NAME, inning_label_class_name)
            current_inning = inning_label_element.text.strip()
        except NoSuchElementException:
            print(f"Warning: Inning label not found in a HalfInning container for game {game_id}.")
        
        joined_text.append(f"\n--- {current_inning} ---")

        # Now, find all "PlayHeader" elements (or a parent that contains both description and scores)
        # Often, a specific parent div wraps the whole play entry. Let's assume PlayHeader is such a parent.
        # If not, you might need to find a common ancestor of play_description_class_name and score classes.
        
        # Searching for PlayHeader__description directly, and then trying to find scores as siblings/cousins.
        # This is a common pattern where the description and scores are peers within a larger play entry div.
        
        play_entries = container.find_elements(By.XPATH, f".//div[contains(@class, 'PlayHeader')]")
        
        if not play_entries:
            # Fallback: if 'PlayHeader' isn't the direct parent, try finding descriptions directly
            # and then trying to find sibling score elements if they exist.
            play_descriptions_only = container.find_elements(By.CLASS_NAME, play_description_class_name)
            for play_desc_element in play_descriptions_only:
                play_text = play_desc_element.text.strip()
                if play_text:
                    if not play_text.endswith(('.', '!', '?')):
                        play_text += "."
                    joined_text.append(play_text)
            continue # Move to next inning container

        for play_entry in play_entries:
            play_text = ""
            away_score = ""
            home_score = ""

            try:
                play_description_element = play_entry.find_element(By.CLASS_NAME, play_description_class_name)
                play_text = play_description_element.text.strip()
            except NoSuchElementException:
                pass # Play description might not be found for every PlayHeader div (e.g., just score updates)

            try:
                away_score_element = play_entry.find_element(By.CLASS_NAME, away_score_class_name.replace(" ", ".")) # Use . for compound class search
                away_score = away_score_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                home_score_element = play_entry.find_element(By.CLASS_NAME, home_score_class_name.replace(" ", ".")) # Use . for compound class search
                home_score = home_score_element.text.strip()
            except NoSuchElementException:
                pass

            if play_text:
                if not play_text.endswith(('.', '!', '?')):
                    play_text += "."
                
                score_info = ""
                if away_score and home_score:
                    score_info = f" ({away_team_name} {away_score}, {home_team_name} {home_score})"
                elif away_score: # In case only one score is updated/shown
                    score_info = f" ({away_team_name} {away_score})"
                elif home_score:
                    score_info = f" ({home_team_name} {home_score})"

                joined_text.append(f"{play_text}{score_info}")
            elif away_score or home_score: # If no play text but score update
                score_info = ""
                if away_score and home_score:
                    score_info = f"Score update: {away_team_name} {away_score}, {home_team_name} {home_score}."
                elif away_score:
                    score_info = f"{away_team_name} score is now {away_score}."
                elif home_score:
                    score_info = f"{home_team_name} score is now {home_score}."
                
                if score_info:
                    joined_text.append(score_info)

    return "\n".join(joined_text).strip()

print("`MLB_play_by_play` function updated to include inning, play descriptions, and associated scores/team names.")

# MLB Line Score Scraper: mlb_line_score
#### This function extracts the game's line score (scores per inning for each team) and formats it into a Markdown table.

In [ ]:
# %%
def mlb_line_score(driver: Web, game_id: str, game_base_url: str, load_url: bool = False) -> str:
    """
    Extracts the MLB game's line score table (scores per inning for each team)
    and formats it into a Markdown table string.

    Args:
        driver (Web): The Selenium WebDriver instance.
        game_id (str): The unique identifier for the MLB game.
        game_base_url (str): The base ESPN MLB URL for game-specific pages.
        load_url (bool): If True, the function will navigate to the game's play-by-play
                         URL before attempting to extract the line score. Set to False if
                         the driver is already on the relevant page (e.g., after `MLB_play_by_play`).

    Returns:
        str: A multi-line string representing the line score in Markdown table format.
             Returns "Line score not available." if data cannot be retrieved.
    """
    print(f"Extracting linescore: {game_id} (URL: {game_base_url}_/gameId/{game_id})")
    play_url = f"{game_base_url}_/gameId/{game_id}"
    
    if load_url:
        driver.get(play_url) # Navigate if explicitly told to load the URL
        
    try:
        # Waits for the 9th inning score of the second team to ensure line score table is fully loaded
        wait_el_text(driver, By.CSS_SELECTOR, "div.LineScore div.Table__ScrollerWrapper table tbody tr:nth-child(2) td:nth-of-type(9)")
    except TimeoutError:
        print(f"Timeout waiting for line score elements for game {game_id} at {game_base_url}. Skipping.")
        return "Line score not available." # Return placeholder if data not found
    
    # Locate the table containing team names (often separate from the main score grid)
    team_table = driver.find_element(By.CSS_SELECTOR, "div.LineScore table:first-of-type")
    team1 = team_table.find_element(By.CSS_SELECTOR,"tr:nth-of-type(2) a.AnchorLink:nth-child(2)").text # Get Team 1 name
    team2 = team_table.find_element(By.CSS_SELECTOR,"tr:nth-of-type(3) a.AnchorLink:nth-child(2)").text # Get Team 2 name
    
    # Locate the main score table (with inning-by-inning scores)
    score_table = driver.find_element(By.CSS_SELECTOR,"div.LineScore div.Table__ScrollerWrapper table")
    team1_scores = score_table.find_elements(By.CSS_SELECTOR,"tbody tr:nth-child(1) td:nth-of-type(-n+9)") # Get Team 1's 9 inning scores
    team2_scores = score_table.find_elements(By.CSS_SELECTOR,"tbody tr:nth-child(2) td:nth-of-type(-n+9)") # Get Team 2's 9 inning scores
    
    # Initialize the Markdown table string with header and separator row
    md_table = "| Team | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |\n| - | - | - | - | - | - | - | - | - | - |\n"
    team1_row = f"| {team1} | " # Start Team 1's row
    team2_row = f"| {team2} | " # Start Team 2's row
    
    # Populate rows with scores for each inning
    for i in range(9): 
        team1_row += team1_scores[i].text + ' | '
        team2_row += team2_scores[i].text + ' | '
    
    # Combine all parts into the final Markdown table string
    return md_table + team1_row + '\n' + team2_row.strip()

print("`mlb_line_score` function defined.")
# %%

# Main Scraper: mlb_play_n_score

In [ ]:
# %%
def mlb_play_n_score():
    """ 
    Orchestrates the extraction of play-by-play scripts and line scores for MLB games.
    Reads game IDs from `GAMEID_FILE`, initializes a headless Firefox WebDriver,
    and then iterates through each game ID to scrape the data.
    The extracted data is appended to a JSON file in the `OUTPUT_DIR`.
    """
    print(f"Make sure your game IDs are listed in the CSV file at `{path.join(GAMEID_FILE)}`")
    
    # Prompt user for an output file name. If empty, generate a timestamped name.
    out_file = input("Press enter for a new output file name or type in an existing file name to append to: ")
    if not out_file:
        out_file = "mlb_play_n_score_" + datetime.datetime.now().strftime("%H%M%S") + ".json"
    
    # Read game IDs from the specified CSV file.
    # Ensure 'gameIds' column is read as string to preserve leading zeros if any.
    df = pd.read_csv(GAMEID_FILE, dtype={"gameIds":"str"})
    game_ids = df["gameIds"].tolist()
    print(f"Initiating extraction for {len(game_ids)} MLB games.")
    
    # Create the output directory if it doesn't already exist.
    makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- Set up the Selenium WebDriver for Firefox ---
    options = webdriver.FirefoxOptions() # Instantiate FirefoxOptions for configuring the browser 
    # Optionally, you might still want to set a specific window size for consistency,
    # even in non-headless mode. If not, the browser will open with its default size.
    options.add_argument("--width=1920") 
    options.add_argument("--height=1080")
    # Initialize Firefox service using GeckoDriverManager to automatically download and manage geckodriver.
    service = FirefoxService(GeckoDriverManager().install())
    # Create the Firefox WebDriver instance with the defined service and options.
    driver = webdriver.Firefox(service=service, options=options)
    
    # --- Loop through each game ID to scrape data ---
    for game_id in game_ids:
        try:
            # Extract play-by-play transcript for the current game
            transcript = MLB_play_by_play(driver, game_id)
            # Extract line score for the current game.
            # load_url=False because MLB_play_by_play already navigated to a relevant page.
            line_score = mlb_line_score(driver, game_id) 
            
            # Prepare the data as a dictionary
            game_data = {
                "game_id": game_id,
                "input": transcript,
                "ground": line_score
            }
            
            # Append the extracted data for the current game to the specified JSON file
            append_json(out_file, [game_data]) # Wrap in a list as append_json expects a list of items
            print(f"Successfully scraped and saved data for game: {game_id}")
        except Exception as e:
            # Catch any exceptions during the scraping process for a specific game
            print(f"⚠️ Error processing game {game_id}: {e}")
            # Optionally, save the game_id to a separate error log for later retry
            # with open(path.join(OUTPUT_DIR, "error_games.txt"), "a") as err_file:
            #     err_file.write(f"{game_id}\n")
    
    # Close the WebDriver when all games have been processed
    driver.quit()
    print("\nMLB Play-by-Play and Line Score scraping completed.")

print("`mlb_play_n_score` function defined.")
# %%

# MLB Pitcher Box Score Scraper: mlb_pitcher_box
#### This function extracts pitcher statistics from the box score page and formats them into a Markdown table.

In [8]:
def mlb_pitcher_box(driver, game_id, game_base_url="https://www.espn.com/mlb/", load_url=False):
    print(f"Extracting pitcher boxscore: {game_id}")
    boxscore_url = game_base_url + "boxscore/_/gameId/" + game_id

    if load_url:
        driver.get(boxscore_url)

    # Wait for pitcher stats to load
    wait_el_text(driver, By.CSS_SELECTOR, ".Boxscore__Category:nth-child(2) .Boxscore__Team:nth-child(2) .Table__Scroller tbody td:last-child")

    # Get all pitcher names and values
    pitchers = driver.find_elements(By.CSS_SELECTOR, ".Boxscore__Category:nth-child(2) .Boxscore__Team .Boxscore__Athlete_Name")
    values = driver.find_elements(By.CSS_SELECTOR, ".Boxscore__Category:nth-child(2) .Boxscore__Team .Table__Scroller tbody tr:not(.Boxscore__Totals) td")

    # Indices to exclude: ER (3), PC-ST (7), ERA (8)
    exclude_indices = [3, 7, 8]
    md_table = "| Pitcher | IP | H | R | BB | K | HR | \n | - | - | - | - | - | - | - | \n "

    for p in range(len(pitchers)):
        pitcher = pitchers[p].text.strip()
        md_table += "| " + pitcher + " | "

        for i in range(9):
            if i in exclude_indices:
                continue
            val = values[i + p * 9].text.strip()
            md_table += val + " | "

        md_table += "\n "

    return md_table


# Main Pitcher Data Scraper: mlb_play_pitcher_box
#### This is the main orchestrator function for scraping play-by-play and pitcher box score data. Similar to mlb_play_n_score, but for pitcher data.

In [ ]:
# %%
def mlb_play_pitcher_box():
    """
    Orchestrates the extraction of play-by-play scripts and pitcher boxscores for MLB games.
    Reads game IDs from `GAMEID_FILE`, initializes a headless Firefox WebDriver,
    and then iterates through each game ID to scrape the data.
    The extracted data is appended to a JSON file in the `OUTPUT_DIR`.
    """
    print(f"Make sure your game IDs are listed in the CSV file at `{path.join(GAMEID_FILE)}`")
    
    # Prompt user for an output file name. If empty, generate a timestamped name.
    out_file = input("Press enter for a new output file name or type in an existing file name to append to: ")
    if not out_file:
        out_file = "mlb_play_pitcher_box_" + datetime.datetime.now().strftime("%H%M%S") + ".json"
    
    # Read game IDs from the specified CSV file.
    # Ensure 'gameIds' column is read as string to preserve leading zeros if any.
    df = pd.read_csv(GAMEID_FILE, dtype={"gameIds":"str"})
    game_ids = df["gameIds"].tolist()
    print(f"Initiating extraction for {len(game_ids)} MLB games.")
    
    # Create the output directory if it doesn't already exist.
    makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- Set up the Selenium WebDriver for Firefox ---
    options = webdriver.FirefoxOptions() # Instantiate FirefoxOptions
    options.add_argument("-headless")  # Run Firefox in headless mode
    
    # Set a specific window size for consistent rendering
    options.add_argument("--width=1920")
    options.add_argument("--height=1080")
    
    # Initialize Firefox service and WebDriver
    service = FirefoxService(GeckoDriverManager().install())
    driver = webdriver.Firefox(service=service, options=options)
    
    # --- Loop through each game ID to scrape data ---
    for game_id in game_ids:
        try:
            # Extract play-by-play transcript for the current game
            transcript = MLB_play_by_play(driver, game_id)
            # Extract pitcher box score for the current game.
            # `load_url=True` is used here because `mlb_pitcher_box` navigates to a different URL
            # than `MLB_play_by_play`.
            boxscore = mlb_pitcher_box(driver, game_id, True) 
            
            # Prepare the data as a dictionary
            game_data = {
                "game_id": game_id,
                "input": transcript,
                "ground": boxscore
            }
            
            # Append the extracted data for the current game to the specified JSON file
            append_json(out_file, game_data) # append_json can take a single dict as data
            print(f"Successfully scraped and saved pitcher boxscore for game: {game_id}")
        except Exception as e:
            # Catch any exceptions during the scraping process for a specific game
            print(f"⚠️ Error processing game {game_id}: {e}")
    
    # Close the WebDriver when all games have been processed
    driver.quit()
    print("\nMLB Play-by-Play and Pitcher Boxscore scraping completed.")

print("`mlb_play_pitcher_box` function defined.")
# %%

# MLB Batter Box Score Scraper: mlb_batter_box
#### This function extracts batter statistics from the box score page and formats them into a Markdown table.

In [10]:
def mlb_batter_box(driver: Web, game_id: str, load_url: bool = False) -> str:
    print(f"Extracting batter boxscore: {game_id}")
    url = f"https://www.espn.com/mlb/boxscore/_/gameId/{game_id}"

    if load_url:
        driver.get(url)

    # Wait until batter name elements load
    wait_el_text(driver, By.CSS_SELECTOR, ".Boxscore__Athlete_Name")

    # Find all teams with batting sections
    team_sections = driver.find_elements(By.CSS_SELECTOR, ".Boxscore__Team")

    md_table = "| Team | Player | Pos | AB | R | H | RBI | HR | BB | K | AVG | OBP | SLG |\n"
    md_table += "| - | - | - | - | - | - | - | - | - | - | - | - | - |\n"

    for team_section in team_sections:
        try:
            team_name = team_section.find_element(By.CSS_SELECTOR, ".TeamTitle__Name").text.replace(" Hitting", "").strip()
        except:
            team_name = "Unknown"

        # Get player rows
        player_rows = team_section.find_elements(By.CSS_SELECTOR, ".Boxscore__Athlete_Name")
        position_tags = team_section.find_elements(By.CSS_SELECTOR, ".Boxscore__Athlete_Position")
        stat_table = team_section.find_elements(By.CSS_SELECTOR, ".Table__Scroller")[0]
        stat_rows = stat_table.find_elements(By.CSS_SELECTOR, "tbody tr:not(.Boxscore__Totals)")

        for i in range(len(player_rows)):
            player_name = player_rows[i].text.strip()
            position = position_tags[i].text.strip() if i < len(position_tags) else "?"

            stat_tds = stat_rows[i].find_elements(By.CSS_SELECTOR, "td")
            if len(stat_tds) < 10:
                continue  # Skip rows with incomplete data

            ab = stat_tds[0].text
            r = stat_tds[1].text
            h = stat_tds[2].text
            rbi = stat_tds[3].text
            hr = stat_tds[4].text
            bb = stat_tds[5].text
            k = stat_tds[6].text
            avg = stat_tds[7].text
            obp = stat_tds[8].text
            slg = stat_tds[9].text

            md_table += f"| {team_name} | {player_name} | {position} | {ab} | {r} | {h} | {rbi} | {hr} | {bb} | {k} | {avg} | {obp} | {slg} |\n"

    return md_table

In [ ]:
# %%
def mlb_batter_box(driver: Web, game_id: str, game_base_url: str, load_url: bool = False) -> str:
    """
    Extracts batter boxscore statistics for all batters in a given MLB game
    and formats them into a detailed Markdown table, including team, player,
    position, and various batting stats (AB, R, H, RBI, HR, BB, K, AVG, OBP, SLG).

    Args:
        driver (Web): The Selenium WebDriver instance.
        game_id (str): The unique identifier for the MLB game.
        game_base_url (str): The base ESPN MLB URL for game-specific pages.
        load_url (bool): If True, the function will navigate to the game's boxscore
                         URL before attempting to extract data.

    Returns:
        str: A multi-line string representing the batter boxscore in Markdown table format.
             Returns "Batter boxscore not available." if data cannot be retrieved.
    """
    print(f"Extracting batter boxscore: {game_id} (URL: {game_base_url}_/gameId/{game_id})")
    url = f"{game_base_url}boxscore/_/gameId/{game_id}"

    if load_url:
        driver.get(url) # Navigate to the boxscore URL if required

    try:
        # Wait until at least one batter name element loads, indicating the table is present
        wait_el_text(driver, By.CSS_SELECTOR, ".Boxscore__Athlete_Name")
    except TimeoutError:
        print(f"Timeout waiting for batter boxscore elements for game {game_id} at {game_base_url}. Skipping.")
        return "Batter boxscore not available." # Return placeholder if data not found

    # Find all sections corresponding to batting teams on the page
    team_sections = driver.find_elements(By.CSS_SELECTOR, ".Boxscore__Team")

    # Initialize the Markdown table string with header and separator for all desired stats
    md_table = "| Team | Player | Pos | AB | R | H | RBI | HR | BB | K | AVG | OBP | SLG |\n"
    md_table += "| - | - | - | - | - | - | - | - | - | - | - | - | - |\n"

    # Iterate through each team's batting section
    for team_section in team_sections:
        try:
            # Extract the team name, removing " Hitting" suffix if present
            team_title_element = team_section.find_element(By.CSS_SELECTOR, ".TeamTitle__Name")
            team_name = team_title_element.text.replace(" Hitting", "").strip()
        except Exception:
            team_name = "Unknown Team" # Fallback if team title element is not found
            print(f"Warning: Could not find team name for a section in game {game_id} at {game_base_url}.")

        # Get all player name elements within this team's section
        player_rows = team_section.find_elements(By.CSS_SELECTOR, ".Boxscore__Athlete")
        # Get all player position elements within this team's section
        position_tags = team_section.find_elements(By.CSS_SELECTOR, ".Boxscore__Athlete_Position")
        
        # Find the statistics table for the current team. Take the first one if multiple are found.
        stat_tables = team_section.find_elements(By.CSS_SELECTOR, ".Table__Scroller")
        if not stat_tables:
            # print(f"Warning: No stat table found for a team section in game {game_id} at {game_base_url}. Skipping team.")
            continue # Skip this team section if no stat table is present
        stat_table = stat_tables[0] # Get the main stat table
        
        # Get all individual player stat rows from the table, excluding any "Totals" row
        stat_rows = stat_table.find_elements(By.CSS_SELECTOR, "tbody tr:not(.Boxscore__Totals)")

        # Iterate through each player identified by name
        for i in range(len(player_rows)):
            player_name = player_rows[i].text.strip()
            # Get player position; default to "?" if position element is not found
            position = position_tags[i].text.strip() if i < len(position_tags) else "?"

            # Ensure there is a corresponding stat row for this player
            if i >= len(stat_rows):
                # print(f"Warning: No corresponding stat row for player {player_name} in game {game_id} at {game_base_url}. Skipping player.")
                continue

            # Get all individual stat cells (td elements) for the current player's row
            stat_tds = stat_rows[i].find_elements(By.CSS_SELECTOR, "td")
            
            # Check if we have the expected number of batting stats (10 columns: AB, R, H, RBI, HR, BB, K, AVG, OBP, SLG)
            if len(stat_tds) < 10:
                # print(f"Warning: Incomplete stats ({len(stat_tds)}/10) for {player_name} in {team_name} (game {game_id}, {game_base_url}). Skipping row.")
                continue  # Skip rows with incomplete data

            # Extract each specific statistic by its column index
            ab = stat_tds[0].text
            r = stat_tds[1].text
            h = stat_tds[2].text
            rbi = stat_tds[3].text
            hr = stat_tds[4].text
            bb = stat_tds[5].text
            k = stat_tds[6].text
            avg = stat_tds[7].text
            obp = stat_tds[8].text
            slg = stat_tds[9].text

            # Append the player's data as a row to the Markdown table
            md_table += f"| {team_name} | {player_name} | {position} | {ab} | {r} | {h} | {rbi} | {hr} | {bb} | {k} | {avg} | {obp} | {slg} |\n"

    return md_table.strip() # Return the full table string, stripped of trailing whitespace

print("`mlb_batter_box` function defined.")
# %%

# Main Batter Data Scraper: mlb_play_batter_box
#### This is the main orchestrator function for scraping play-by-play and batter box score data.

In [ ]:
# %%
def mlb_play_batter_box():
    """
    Orchestrates the extraction of play-by-play scripts and batter boxscores for MLB games.
    Reads game IDs from `GAMEID_FILE`, initializes a headless Firefox WebDriver,
    and then iterates through each game ID to scrape the data.
    The extracted data is appended to a JSON file in the `OUTPUT_DIR`.
    """
    print(f"Make sure your game IDs are listed in the CSV file at `{path.join(GAMEID_FILE)}`")
    
    # Prompt user for an output file name. If empty, generate a timestamped name.
    out_file = input("Press enter for a new output file name or type in an existing file name to append to: ")
    if not out_file:
        out_file = "mlb_play_batter_box_" + datetime.datetime.now().strftime("%H%M%S") + ".json"

    # Read game IDs from the specified CSV file.
    # Ensure 'gameIds' column is read as string to preserve leading zeros if any.
    df = pd.read_csv(GAMEID_FILE, dtype={"gameIds": "str"})
    game_ids = df["gameIds"].tolist()
    print(f"Initiating extraction for {len(game_ids)} MLB games.")

    # Create the output directory if it doesn't already exist.
    makedirs(OUTPUT_DIR, exist_ok=True)

    # --- Set up the Selenium WebDriver for Firefox ---
    options = webdriver.FirefoxOptions() # Instantiate FirefoxOptions
    options.add_argument("-headless")  # Run Firefox in headless mode
    
    # Set a specific window size for consistent rendering
    options.add_argument("--width=1920")
    options.add_argument("--height=1080")
    
    # Initialize Firefox service and WebDriver
    service = FirefoxService(GeckoDriverManager().install())
    driver = webdriver.Firefox(service=service, options=options)

    # --- Loop through each game ID to scrape data ---
    for game_id in game_ids:
        try:
            # Extract play-by-play transcript for the current game
            transcript = MLB_play_by_play(driver, game_id)
            # Extract batter box score for the current game.
            # `load_url=True` is used here because `mlb_batter_box` navigates to a different URL
            # than `MLB_play_by_play`.
            batter_box = mlb_batter_box(driver, game_id, True)
            
            # Prepare the data as a list containing a single dictionary
            game_data = {
                "game_id": game_id,
                "input": transcript,
                "ground": batter_box
            }
            
            # Append the extracted data for the current game to the specified JSON file
            append_json(out_file, [game_data]) # append_json expects a list for extension
            print(f"Successfully scraped and saved batter boxscore for game: {game_id}")
        except Exception as e:
            # Catch any exceptions during the scraping process for a specific game
            print(f"⚠️ Error processing game {game_id}: {e}")

    # Close the WebDriver when all games have been processed
    driver.quit()
    print("\nMLB Play-by-Play and Batter Boxscore scraping completed.")

print("`mlb_play_batter_box` function defined.")
# %%

# Main Execution Block:
#### This cell serves as the entry point when the script is run. It presents a menu to the user to choose which scraping task to perform.

In [15]:
# %%
if __name__ == "__main__":
    print("Welcome to the MLB Data Scraper!")
    print("Starting comprehensive scraping for all tasks and languages for specified dates.")

    # Define the dates you want to scrape. Format: YYYYMMDD.
    # You can add multiple dates to scrape data for a range of days.
    dates_to_scrape = []
    #assuming 15 games a day, we can add dates from 20250622 backwards to num_samples//15
    num_samples = 1000 # Total number of samples to scrape
    for i in range(num_samples // 15 + 2): #add an extra day to ensure we cover the range
        date = (datetime.datetime(2025, 6, 25) - datetime.timedelta(days=i)).strftime("%Y%m%d")
        dates_to_scrape.append(date)
    # Create the output directory if it doesn't already exist.
    makedirs(OUTPUT_DIR, exist_ok=True)

    # --- Set up the Selenium WebDriver for Firefox (Non-headless as requested) ---
    options = webdriver.FirefoxOptions() # Instantiate FirefoxOptions for browser configuration
    # options.add_argument("-headless") # REMOVED: This line would make the browser invisible.
                                     # By commenting it out, the browser window will be visible.
    options.add_argument("--width=1920") # Set desired window width for consistent rendering
    options.add_argument("--height=1080") # Set desired window height
    
    # Initialize Firefox service using GeckoDriverManager to automatically download and manage geckodriver.
    service = FirefoxService(GeckoDriverManager().install())
    # Create the Firefox WebDriver instance. This driver will be reused for all scraping tasks.
    driver = webdriver.Firefox(service=service, options=options)

    try:
        # Iterate through each defined language (e.g., "en" for English, "es" for Spanish)
        for lang_code, lang_info in LANGUAGES.items():
            game_base_url = lang_info['game_base_url']
            scoreboard_base_url = lang_info['scoreboard_base_url']
            date_param = lang_info['date_param']

            print(f"\n--- Processing language: {lang_code.upper()} ---")
            samples_processed = 0 # Initialize a counter to track the number of samples processed
            # Iterate through each date specified in `dates_to_scrape`
            for date_str in dates_to_scrape:
                print(f"\n--- Processing date: {date_str} for {lang_code.upper()} ---")

                # 1. Get all game IDs for the current date and language from the scoreboard page
                game_ids = get_game_ids_for_date(driver, scoreboard_base_url, date_param, date_str)

                if not game_ids:
                    print(f"No games found for {date_str} in {lang_code.upper()}. Skipping to next date/language.")
                    continue # If no games, move to the next date or language

                # Define output file names for the current date and language.
                # Data for each type of scrape (play-by-play/linescore, pitcher box, batter box)
                # will be appended to separate files for organization.
                play_n_score_file = f"mlb_play_n_score_{lang_code}_{date_str}.json"
                pitcher_box_file = f"mlb_pitcher_box_{lang_code}_{date_str}.json"
                batter_box_file = f"mlb_batter_box_{lang_code}_{date_str}.json"

                # 2. Loop through each game ID found and scrape detailed data
                for i, game_id in enumerate(game_ids):
                    print(f"[{i+1}/{len(game_ids)}] Processing game ID: {game_id}")
                    # Prepare base data for JSON output, which will be extended for each specific scrape
                    current_game_data = {"game_id": game_id, "language": lang_code, "date": date_str}

                    # --- Scrape Play-by-Play and Line Score ---
                    try:
                        # MLB_play_by_play navigates to the play-by-play page
                        transcript = MLB_play_by_play(driver, game_id, game_base_url)
                        # mlb_line_score can then scrape from the current page, no need to reload
                        line_score = mlb_line_score(driver, game_id, game_base_url, load_url=False)
                        # Append the combined data to the play-by-play/linescore JSON file
                        append_json(play_n_score_file, [{**current_game_data, "input": transcript, "ground": line_score}])
                        print(f"  - Scraped play-by-play & linescore for {game_id}")
                    except Exception as e:
                        print(f"  ❌ Error scraping play-by-play/linescore for {game_id}: {e}")
                        # Optionally append an error entry for this game/task
                        # append_json(play_n_score_file, [{**current_game_data, "input": "Error", "ground": "Error", "error_detail": str(e)}])

                    # --- Scrape Pitcher Box Score ---
                    try:
                        # Explicitly load URL for boxscore as it's a different page from play-by-play
                        pitcher_box = mlb_pitcher_box(driver, game_id, game_base_url, load_url=True)
                        # Append to pitcher boxscore JSON file. 'input' field is omitted here if not directly related to a transcript.
                        append_json(pitcher_box_file, [{**current_game_data, "ground": pitcher_box}])
                        print(f"  - Scraped pitcher boxscore for {game_id}")
                    except Exception as e:
                        print(f"  ❌ Error scraping pitcher boxscore for {game_id}: {e}")
                        # append_json(pitcher_box_file, [{**current_game_data, "ground": "Error", "error_detail": str(e)}])

                    # --- Scrape Batter Box Score ---
                    try:
                        # Explicitly load URL for boxscore again for robustness, ensuring we're on the correct page.
                        batter_box = mlb_batter_box(driver, game_id, game_base_url, load_url=True)
                        # Append to batter boxscore JSON file.
                        append_json(batter_box_file, [{**current_game_data, "ground": batter_box}])
                        print(f"  - Scraped batter boxscore for {game_id}")
                    except Exception as e:
                        print(f"  ❌ Error scraping batter boxscore for {game_id}: {e}")
                        # append_json(batter_box_file, [{**current_game_data, "ground": "Error", "error_detail": str(e)}])
                samples_processed += len(game_ids) # Update the total samples processed for this language/date
                print(f"\nCompleted scraping for {len(game_ids)} games on {date_str} in {lang_code.upper()}.")
                #restart browser every day to prevent memory leaks
                print("Restarting browser to prevent memory leaks...")
                driver.quit()
                driver = webdriver.Firefox(service=service, options=options)
        total_games = len(dates_to_scrape) * len(game_ids)*len(LANGUAGES) 
        print(f"\nCompleted scraping for {total_games} games.")                       
    except Exception as e:
        # Catch any unhandled exceptions from the main loop
        print(f"An unhandled critical error occurred during the main scraping process: {e}")
    finally:
        # Ensure the WebDriver is closed gracefully, even if errors occur, to free up resources.
        driver.quit()
        print("\nAll MLB Data Scraping tasks for all languages and dates completed.")
# %%

Welcome to the MLB Data Scraper!
Starting comprehensive scraping for all tasks and languages for specified dates.

--- Processing language: ENGLISH ---

--- Processing date: 20250625 for ENGLISH ---
Attempting to get game IDs from: https://www.espn.com/mlb/scoreboard/_/date/20250625
Found 15 games for 20250625 at https://www.espn.com/mlb/scoreboard/.
[1/15] Processing game ID: 401696101
Extracting play-by-play: 401696101 (URL: https://www.espn.com/mlb/playbyplay/_/gameId/401696101)
Detected Teams: Away - Braves, Home - Mets
Extracting linescore: 401696101 (URL: https://www.espn.com/mlb/_/gameId/401696101)
File 'mlb_play_n_score_english_20250625.json' not found or invalid JSON. Creating new file/content.
  - Scraped play-by-play & linescore for 401696101
Extracting pitcher boxscore: 401696101
File 'mlb_pitcher_box_english_20250625.json' not found or invalid JSON. Creating new file/content.
  - Scraped pitcher boxscore for 401696101
Extracting batter boxscore: 401696101 (URL: https://www